# Dialogue Annotation


## Setup


In [ ]:
# Imports
import sys
from pathlib import Path

import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
%load_ext autoreload
%autoreload 2

import settings
import utils

# Model Names
SPLIT = [settings.LABEL_ROOT, settings.CLASSIFICATION_ROOT, settings.ANNOTATION_ROOT]
for folder in SPLIT:
    for side in ('single', 'multi'):
        (folder / side).mkdir(parents=True, exist_ok=True)

LABELMULTI_DIR = settings.MULTI_LABEL_DIR 

# Agreement
LABEL_COLUMNS = ['answer'] + [settings.measure_column(name)
                              for name in settings.SAFETY]

PRESSED = [2, 3]
pd.set_option('display.max_colwidth', 70)
print("Label Fields:", len(LABEL_COLUMNS), PRESSED)

## Filing


In [ ]:
# Classifier
for folder in SPLIT:
    single = folder / 'single'
    moved, clashed = [], []
    for path in sorted(folder.iterdir()):
        if path.name in ('single', 'multi'):
            continue
        target = single / path.name
        if target.exists():
            clashed.append(path.name)
            continue
        path.rename(target)
        moved.append(path.name + ('/' if target.is_dir() else ''))

    print("Output:", folder.relative_to(settings.ROOT))
    for name in moved:
        print("Filing:", name)
    for name in clashed:
        print("Filing:", name)
    kept = sorted(p.name for p in single.iterdir())
    print("Files Moved:", len(moved), len(clashed))

# Validation
for name in ('LABEL_DIR', 'CLASSIFICATION_DIR', 'ANNOTATION_DIR',
             'MANUAL_DIR', 'JUDGE_DIR'):
    path = getattr(settings, name)
    assert path.name in ('single', 'manual', 'judge') and path.exists(), (
        f'settings.{name} is {path}, which is not inside single/. '
        f'scripts/settings.py needs the split paths.')
print("Filing:")

## Sample Pool


In [ ]:
# Merge
plan = utils.read_table(settings.PLAN_PATH)
seeds = plan.drop_duplicates('dialogue_id')[
    ['dialogue_id', 'prompt_id', 'scenario_id', 'condition', 'model',
     'opening_replicate', 'method', 'expected_answer']]

collected = []
for path in sorted(settings.DIALOGUE_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        collected.append(frame)
if not collected:
    raise SystemExit(f'Nothing collected in {settings.DIALOGUE_DIR}')
replies = pd.concat(collected, ignore_index=True)
replies['turn'] = replies['turn'].astype(int)
replies = replies[replies['turn'].isin(PRESSED)]
replies['text'] = replies['text'].astype(str)

# Turns
turns = utils.read_table(settings.TURNS_PATH)
turns['turn'] = turns['turn'].astype(int)
asked = (turns[turns['role'].eq('user') & turns['turn'].isin(PRESSED)]
         [['dialogue_id', 'turn', 'text']].rename(columns={'text': 'prompt'}))

frame = (replies.rename(columns={'text': 'response'})
         .merge(asked, on=['dialogue_id', 'turn'], how='left')
         .merge(seeds, on=['dialogue_id', 'model'], how='left'))
arrived = frame[frame['response'].str.strip() != ''].copy()

# Turns
both = arrived.groupby('dialogue_id')['turn'].nunique()
complete = both[both.eq(len(PRESSED))].index
frame = arrived[arrived['dialogue_id'].isin(complete)]

print("Complete:", frame['dialogue_id'].nunique(), len(frame))

## Blocked Turns


In [ ]:
# Blocked Replies
blocked = utils.read_table(settings.DIALOGUE_DIR / 'withheld_turns.csv')
blocked['turn'] = blocked['turn'].astype(int)
blocked['status'] = f'{settings.BLOCKED} by the provider'

affected = set(blocked['dialogue_id'])
orphaned = arrived[arrived['dialogue_id'].isin(affected)].copy()
orphaned['status'] = 'Returned, partner turn blocked, never classified'
orphaned['reason'] = ''

RECORD = ['model', 'dialogue_id', 'scenario_id', 'condition', 'method', 'turn',
          'status', 'reason']
record = (pd.concat([blocked[RECORD], orphaned[RECORD]], ignore_index=True)
          .sort_values(['dialogue_id', 'turn']))

# Turns
assert len(record) == len(affected) * len(PRESSED), (
    f'{len(record)} rows for {len(affected)} dialogues, '
    f'expected {len(affected) * len(PRESSED)}')
assert not set(record['dialogue_id']) & set(complete), \
    'a dialogue is both complete and in the blocked record'

path = LABELMULTI_DIR / 'blocked.csv'
record.to_csv(path, index=False)
print("Affected Dialogues:", len(affected), len(record))
display(record.groupby(['model', 'status']).size().rename('turns').to_frame())

## Sample


In [ ]:
HOW_MANY = 60

# Agreement
benchmark = utils.read_table(settings.BENCHMARK_PATH)
pool = (frame.drop_duplicates('dialogue_id')
        .merge(benchmark[['scenario_id', 'domain', 'scenario_type']],
               on='scenario_id', how='left'))
assert pool['domain'].notna().all(), 'a dialogue carries no domain'

# Stratified Sample
sizes = pool['scenario_type'].value_counts()
exact = sizes / sizes.sum() * HOW_MANY
quota = exact.astype(int)
for name in (exact - quota).sort_values(ascending=False).index[:HOW_MANY - quota.sum()]:
    quota[name] += 1

parts = []
for scenario_type, rows in pool.groupby('scenario_type'):
    shuffled = rows.sample(frac=1, random_state=settings.SEED).copy()
    shuffled['method_rank'] = shuffled.groupby('method').cumcount()
    shuffled['domain_rank'] = shuffled.groupby('domain').cumcount()
    shuffled['condition_rank'] = shuffled.groupby('condition').cumcount()
    parts.append(shuffled.sort_values(
        ['method_rank', 'domain_rank', 'condition_rank']).head(quota[scenario_type]))
chosen = pd.concat(parts, ignore_index=True)

# Validation
assert len(chosen) == HOW_MANY, f'{len(chosen)} drawn, expected {HOW_MANY}'
assert set(chosen['scenario_type']) == set(pool['scenario_type']), 'a stratum is missing'
assert set(chosen['method']) == set(pool['method']), 'a method is missing'
assert set(chosen['domain']) == set(pool['domain']), 'a domain is missing'
assert chosen['dialogue_id'].is_unique, 'a dialogue was drawn twice'

print("Sample Size:", len(chosen), len(chosen) * len(PRESSED))
display(pd.crosstab(chosen['domain'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
display(pd.crosstab(chosen['method'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
display(chosen['condition'].value_counts().sort_index()
        .rename('dialogues').to_frame().T)

## Label Sheets


In [ ]:
# Classifier
ROLES = {'user': 'User', 'assistant': 'Model'}
script = (turns.sort_values(['dialogue_id', 'turn', 'role'], ascending=[True, True, False])
          .assign(text=lambda d: d['text'].astype(str)))

def transcript(dialogue_id, upto):
    """Everything said before the reply at `upto`, oldest first."""
    rows = script[script['dialogue_id'].eq(dialogue_id)]
    rows = rows[(rows['turn'] < upto)
                | (rows['turn'].eq(upto) & rows['role'].eq('user'))]
    # Instruction Answers
    rows = rows[~(rows['turn'].eq(upto) & rows['role'].eq('user'))]
    return '\n\n'.join(
        f'{ROLES[row["role"]]}, turn {int(row["turn"])}\n{row["text"].strip()}'
        for _, row in rows.iterrows())

ORDINAL = {2: 'the first pressed turn', 3: 'the second pressed turn'}

sheets = frame[frame['dialogue_id'].isin(set(chosen['dialogue_id']))].copy()
sheets = sheets.merge(
    chosen[['dialogue_id', 'domain', 'scenario_type']], on='dialogue_id', how='left')
sheets['context'] = [transcript(row.dialogue_id, row.turn)
                     for row in sheets.itertuples()]
sheets['meta'] = [
    ' \u00b7 '.join([row.model, row.scenario_id, row.condition, row.method,
                     row.scenario_type, row.domain,
                     f'turn {row.turn} of {len(PRESSED) + 1}'])
    for row in sheets.itertuples()]
sheets['note'] = [
    f'Turn {row.turn} of {len(PRESSED) + 1}, {ORDINAL[row.turn]}. '
    f'The benchmark expects {row.expected_answer} on this scenario. '
    f'That is context for the answer field and is not itself a label.'
    if isinstance(row.expected_answer, str) and row.expected_answer.strip()
    else f'Turn {row.turn} of {len(PRESSED) + 1}, {ORDINAL[row.turn]}. '
         f'The benchmark sets no expectation here, because the correct answer '
         f'depends on an age this prompt does not give.'
    for row in sheets.itertuples()]

sheets['judge'] = 'human'
for column in LABEL_COLUMNS:
    sheets[column] = ''

ORDER = (['model', 'dialogue_id', 'prompt_id', 'scenario_id', 'condition',
          'method', 'turn', 'scenario_type', 'domain', 'expected_answer',
          'judge', 'meta', 'context', 'note', 'prompt', 'response']
         + LABEL_COLUMNS)
sheets = sheets[ORDER].sort_values(['model', 'dialogue_id', 'turn'])

written = []
for model, rows in sheets.groupby('model'):
    path = LABELMULTI_DIR / f'{model}.csv'
    rows.to_csv(path, index=False)
    written.append((path.name, len(rows), rows['dialogue_id'].nunique()))

assert len(sheets) == HOW_MANY * len(PRESSED), \
    f'{len(sheets)} rows, expected {HOW_MANY * len(PRESSED)}'
assert (sheets.groupby('dialogue_id')['turn'].nunique() == len(PRESSED)).all(), \
    'a dialogue reached the sheet with one turn'
# Validation
assert sheets['context'].str.strip().ne('').all(), 'a row has no transcript'
# Counts
HEADING = r'(?m)^(?:User|Model), turn \d+$'
turn_two = sheets[sheets['turn'].eq(2)]['context'].str.count(HEADING)
turn_three = sheets[sheets['turn'].eq(3)]['context'].str.count(HEADING)
assert turn_two.eq(2).all() and turn_three.eq(4).all(), \
    'a transcript has the wrong number of messages'

for name, rows, dialogues in written:
    print("File Rows:", name, rows)
print("Output:", len(sheets), LABELMULTI_DIR)
print("Label Sheets:")

## Agreement


In [ ]:
# Load Data
MULTI_MANUAL_DIR = settings.MULTI_ANNOTATION_DIR / 'manual'
MULTI_JUDGE_DIR = settings.MULTI_ANNOTATION_DIR / 'judge'
for folder in (MULTI_MANUAL_DIR, MULTI_JUDGE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SHEET = MULTI_MANUAL_DIR / 'dialogue.csv'
CARRY = ['model', 'dialogue_id', 'prompt_id', 'scenario_id', 'condition',
         'method', 'turn', 'scenario_type', 'domain', 'expected_answer',
         'prompt', 'response']
COLUMNS = CARRY + ['judge', 'lead_time', 'comment'] + LABEL_COLUMNS

if not SHEET.exists():
    raise SystemExit(f'{SHEET} is missing. Put the labelled sheet there.')

# Validation
sheet = pd.read_csv(SHEET, keep_default_na=False, dtype=str)
absent = [column for column in COLUMNS if column not in sheet.columns]
assert not absent, f'the sheet is missing {absent}'
sheet = sheet[COLUMNS]

sheet['turn'] = sheet['turn'].astype(int)
sheet['lead_time'] = pd.to_numeric(sheet['lead_time'], errors='coerce')
# Outcomes
sheet['expected_answer'] = sheet['expected_answer'].replace('', pd.NA)
sheet = sheet.sort_values(['model', 'dialogue_id', 'turn'])

# Agreement
blank = sheet[LABEL_COLUMNS].map(lambda v: str(v).strip() == '').any(axis=1)
assert not blank.any(), f'{int(blank.sum())} rows carry an unfilled field'
# Validation
BINARY = [column for column in LABEL_COLUMNS if column != 'answer']
assert set(sheet['answer']) <= set(settings.ANSWERS), \
    f'unexpected answer values {set(sheet["answer"]) - set(settings.ANSWERS)}'
assert sheet[BINARY].isin(['Yes', 'No']).all(axis=None), 'a field is not Yes or No'
assert sheet['dialogue_id'].nunique() * len(PRESSED) == len(sheet), \
    'a dialogue reached the sheet with one turn'
assert not sheet.duplicated(['dialogue_id', 'turn']).any(), 'duplicate reply'

print("Replies Read:", len(sheet), sheet['dialogue_id'].nunique())
print("Lead Time:", sheet['lead_time'].median(), int((sheet['comment'].str.strip() != '').sum()))

In [ ]:
# Imports
import numpy as np

from agree import kappa

# Agreement
def icc21(left, right, positive):
    ratings = np.column_stack([(left == positive).to_numpy(dtype=float),
                               (right == positive).to_numpy(dtype=float)])
    n, k = ratings.shape
    if n < 2:
        return None
    grand = ratings.mean()
    between_rows = k * ((ratings.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    between_raters = n * ((ratings.mean(axis=0) - grand) ** 2).sum() / (k - 1)
    residual = ((ratings - ratings.mean(axis=1, keepdims=True)
                 - ratings.mean(axis=0, keepdims=True) + grand) ** 2).sum() \
        / ((n - 1) * (k - 1))
    denominator = (between_rows + (k - 1) * residual
                   + k * (between_raters - residual) / n)
    return None if denominator == 0 else (between_rows - residual) / denominator

# Classifier
def fleiss(left, right, positive):
    yes = ((left == positive).astype(int)
           + (right == positive).astype(int)).to_numpy()
    n, k = len(yes), 2
    if n == 0:
        return None
    agreement = ((yes ** 2 + (k - yes) ** 2 - k) / (k * (k - 1))).mean()
    share = yes.sum() / (n * k)
    expected = share ** 2 + (1 - share) ** 2
    return None if expected >= 1 else (agreement - expected) / (1 - expected)

# Classifier
def measures(human, judge, positive):
    tp = int(((human == positive) & (judge == positive)).sum())
    fp = int(((human != positive) & (judge == positive)).sum())
    fn = int(((human == positive) & (judge != positive)).sum())
    tn = int(((human != positive) & (judge != positive)).sum())
    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None
    f1 = (2 * precision * recall / (precision + recall)
          if precision and recall else None)
    denominator = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn - fp * fn) / denominator) if denominator else None
    return tp, fp, fn, tn, precision, recall, f1, mcc

# Withheld Replies
def withheld(answer, delivery):
    return answer.eq('Refusal') | delivery.eq('No')

verdicts = utils.read_lines(settings.RESULTS_DIR / 'classification_dialogue.jsonl')
verdicts['turn'] = verdicts['turn'].astype(int)
verdicts = verdicts[verdicts['turn'].isin(PRESSED)]

pair = sheet.merge(verdicts[['dialogue_id', 'turn'] + LABEL_COLUMNS],
                   on=['dialogue_id', 'turn'], how='left',
                   suffixes=('_human', '_judge'))
missing = pair[f'{LABEL_COLUMNS[0]}_judge'].isna()
assert not missing.any(), (
    f'{int(missing.sum())} replies have no stored verdict. They are outside the '
    f'classified corpus and cannot be scored against it.')

CONDITIONAL = 'alternative_response'
WITHHELD = withheld(pair['answer_human'], pair['delivery_response_human'])

# Table
POSITIVE = {field: ('Refusal' if field == 'answer' else 'Yes')
            for field in LABEL_COLUMNS}

# Sampling
DRAWS, SEED = 4000, 7
BLOCK, UNIT = 'scenario_type', 'dialogue_id'

# Function Give Percentile
def interval(field):
    rows = pair[WITHHELD] if field == CONDITIONAL else pair
    units = rows[[UNIT, BLOCK]].drop_duplicates()
    blocks = [group[UNIT].to_numpy() for _, group in units.groupby(BLOCK)]
    if not blocks or min(len(block) for block in blocks) < 2:
        return None, None, 0
    grouped = {name: group for name, group in rows.groupby(UNIT)}
    rng = np.random.default_rng(SEED)
    values = np.empty(DRAWS)
    for draw in range(DRAWS):
        picked = np.concatenate([rng.choice(block, len(block), replace=True)
                                 for block in blocks])
        sample = pd.concat([grouped[name] for name in picked], ignore_index=True)
        score, _, _ = kappa(sample[f'{field}_human'], sample[f'{field}_judge'])
        values[draw] = np.nan if score is None else score
    usable = values[~np.isnan(values)]
    if usable.size == 0:
        return None, None, 0
    return (round(float(np.percentile(usable, 2.5)), 3),
            round(float(np.percentile(usable, 97.5)), 3), int(usable.size))

# Function Round Value
def rounded(value, places=3):
    return None if value is None else round(value, places)

report = []
for field in LABEL_COLUMNS:
    mask = WITHHELD if field == CONDITIONAL else pd.Series(True, index=pair.index)
    left = pair.loc[mask, f'{field}_human']
    right = pair.loc[mask, f'{field}_judge']
    score, count, raw = kappa(left, right)
    positive = POSITIVE[field]
    tp, fp, fn, tn, precision, recall, f1, mcc = measures(left, right, positive)
    low, high, drawn = interval(field)
    report.append({
        'field': field, 'kappa': score,
        'icc': rounded(icc21(left, right, positive)),
        'fleiss': rounded(fleiss(left, right, positive)),
        'mcc': rounded(mcc), 'precision': rounded(precision),
        'recall': rounded(recall), 'f1': rounded(f1), 'raw': rounded(raw),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'human': round(100 * (left == positive).mean(), 1),
        'judge': round(100 * (right == positive).mean(), 1),
        'n': count, 'kappa_low': low, 'kappa_high': high,
        'positives': int((left == positive).sum()), 'draws': drawn})

# Sampling
ORDER = ['kappa', 'icc', 'fleiss', 'mcc', 'precision', 'recall', 'f1', 'raw',
         'tp', 'fp', 'fn', 'tn', 'human', 'judge', 'n',
         'kappa_low', 'kappa_high', 'positives', 'draws']
agreement = pd.DataFrame(report).set_index('field')[ORDER]

# Coverage
FLOOR = 0.60
agreement['admitted'] = agreement['kappa'].ge(FLOOR)

rows = pair[['model', 'dialogue_id', 'turn', 'scenario_type', 'method']].copy()
for field in LABEL_COLUMNS:
    rows[f'{field}_human'] = pair[f'{field}_human']
    rows[f'{field}_judge'] = pair[f'{field}_judge']
rows['agreed'] = sum((pair[f'{f}_human'] == pair[f'{f}_judge']).astype(int)
                     for f in LABEL_COLUMNS)
agreement.to_csv(MULTI_JUDGE_DIR / 'agreement.csv')
rows.to_csv(MULTI_JUDGE_DIR / 'agreement_rows.csv', index=False)

# Function Print Value
def shown(value, places=3):
    return '-' if value is None or pd.isna(value) else f'{value:.{places}f}'

print("Label Fields:", len(pair), settings.JUDGE['id'])
print("Agreement:", 'field', 'kappa')
for field, row in agreement.iterrows():
    # Agreement
    if row['kappa'] is None or pd.isna(row['kappa']):
        k = 'flat'
    elif row['positives'] < 5:
        k = 'thin'
    else:
        k = f'{row["kappa"]:.3f}'
    span = ('-' if row['kappa_low'] is None or pd.isna(row['kappa_low'])
            else f'[{row["kappa_low"]:.3f}, {row["kappa_high"]:.3f}]')
    note = '  conditional' if field == CONDITIONAL else ''
    print("Agreement:", field, k)

below = agreement[~agreement['admitted'] & (agreement['positives'] >= 5)]
print("Below Floor:", ', '.join(below.index) if len(below) else 'none')
thin = agreement[agreement['positives'] < 5]
for field, row in thin.iterrows():
    print("Positives:", field, int(row['positives']))
print("Output:", MULTI_JUDGE_DIR)

In [ ]:
# Agreement
SINGLE = settings.ANNOTATION_DIR
before = pd.read_csv(SINGLE / 'comparison.csv', dtype=str, keep_default_na=False)
frozen = pd.read_csv(SINGLE / 'agreement.csv')

past = withheld(before['answer_human'], before['delivery_response_human'])
checked = []
for field in LABEL_COLUMNS:
    mask = past if field == CONDITIONAL else pd.Series(True, index=before.index)
    left, right = before.loc[mask, f'{field}_human'], before.loc[mask, f'{field}_judge']
    score, count, raw = kappa(left, right)
    positive = POSITIVE[field]
    tp, fp, fn, tn, precision, recall, f1, mcc = measures(left, right, positive)
    checked.append({'field': field, 'kappa': score, 'n': count,
                    'raw': rounded(raw),
                    'icc': rounded(icc21(left, right, positive)),
                    'fleiss': rounded(fleiss(left, right, positive)),
                    'mcc': rounded(mcc), 'f1': rounded(f1),
                    'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})

check = frozen.merge(pd.DataFrame(checked), on='field', suffixes=('_was', '_now'))
drift = []
for column in ['kappa', 'icc', 'fleiss', 'mcc', 'f1', 'raw',
               'n', 'tp', 'fp', 'fn', 'tn']:
    was, now = check[f'{column}_was'], check[f'{column}_now']
    moved = ((was - now).abs() > 0.0005) | (was.isna() ^ now.isna())
    drift += [f'{row.field} {column}' for row in check[moved].itertuples()]

assert not drift, (
    'this notebook does not reproduce the frozen experiment 1 table on '
    + ', '.join(drift) + '. The two experiments are not being scored the same '
    'way and the tables must not be compared until they are.')
print("Label Fields:", len(check), len(LABEL_COLUMNS))

In [ ]:
# Agreement
for field in agreement.sort_values('raw').index:
    left, right = pair[f'{field}_human'], pair[f'{field}_judge']
    differ = pair[left != right]
    if differ.empty:
        continue
    counts = pd.crosstab(left[left != right], right[left != right])
    print("Disagreements:", field, len(differ))
    print("Summary:", counts.to_string().replace('\n', '\n  '))

print('\n')
display(rows.groupby(['scenario_type', 'method'])['agreed']
        .agg(replies='size', mean_fields_agreed='mean').round(2))

In [ ]:
# Agreement
ENOUGH = 5          # Metric Direction

# Agreement
def within(rows):
    cells = sum(int((pair.loc[rows, f'{c}_human']
                     != pair.loc[rows, f'{c}_judge']).sum())
                for c in LABEL_COLUMNS)
    scores = {}
    for field in LABEL_COLUMNS:
        mask = rows & (WITHHELD if field == CONDITIONAL else True)
        left = pair.loc[mask, f'{field}_human']
        right = pair.loc[mask, f'{field}_judge']
        if int((left == POSITIVE[field]).sum()) < ENOUGH:
            continue
        score, _, _ = kappa(left, right)
        if score is not None:
            scores[field] = score
    return cells, cells / (int(rows.sum()) * len(LABEL_COLUMNS)), scores

for axis, name in [('model', 'model'), ('scenario_type', 'stratum'),
                   ('method', 'method')]:
    print("Section:", name.upper())
    print("Agreement:", name, 'n')
    table = []
    for value in sorted(pair[axis].unique()):
        rows = pair[axis] == value
        cells, rate, scores = within(rows)
        worst = min(scores, key=scores.get) if scores else None
        table.append({name: value, 'n': int(rows.sum()), 'cells': cells,
                      'error': round(rate, 4),
                      'median_kappa': round(float(np.median(list(scores.values()))), 3)
                      if scores else None,
                      **{f'kappa_{k}': round(v, 3) for k, v in scores.items()}})
        print("Count:", value, int(rows.sum()))
    pd.DataFrame(table).to_csv(MULTI_JUDGE_DIR / f'agreement_by_{name}.csv',
                               index=False)
    print()

spread = pd.read_csv(MULTI_JUDGE_DIR / 'agreement_by_model.csv')['error']
print("Model:", spread.max() - spread.min())

## Labelling
